# BioSkills Lab — AI/ML Capstone
## Did the Foundation Model Actually Help?

**Course:** AI for Genomics in the Foundation Model Era  
**Dataset:** scIB PBMC Benchmark (Luecken et al., Nature Methods 2022)  
**Task:** Cell type annotation — comparing PCA, scVI, and Geneformer  

[![BioSkills Lab](https://img.shields.io/badge/BioSkills-Lab-38bdf8)](https://bioskillslab.dev)

---

### Runtime recommendation
- For PCA and scVI: CPU runtime is fine
- For Geneformer: change to **GPU runtime** (Runtime > Change runtime type > T4 GPU)

### What you will build
Three representations of the same cells, one classifier for all three, one honest comparison table.

In [ ]:
# Step 0: Install dependencies
!pip install -q scanpy scvi-tools anndata scikit-learn umap-learn loompy
!pip install -q geneformer  # may take a few minutes

In [ ]:
import scanpy as sc
import scvi
import numpy as np
import pandas as pd
import torch
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt

print(f'scanpy: {sc.__version__}')
print(f'scvi-tools: {scvi.__version__}')
print(f'GPU available: {torch.cuda.is_available()}')

## Step 1: Load the Dataset

In [ ]:
import urllib.request

url  = 'https://figshare.com/ndownloader/files/24539842'
path = 'pbmc_scib.h5ad'

print('Downloading scIB PBMC benchmark dataset (~150 MB)...')
urllib.request.urlretrieve(url, path)
print('Done.')

adata = sc.read_h5ad(path)
print(adata)
print('\nCell types:')
print(adata.obs['cell_type'].value_counts())
print('\nDonors (batches):')
print(adata.obs['batch'].value_counts())

## Step 2: Preprocessing

In [ ]:
# Store raw counts for scVI
adata.layers['counts'] = adata.X.copy()

# Highly variable genes
sc.pp.highly_variable_genes(adata, n_top_genes=2000,
                              subset=False, flavor='seurat_v3',
                              batch_key='batch')

# Normalise and log-transform for PCA
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.scale(adata, max_value=10)

# Encode labels
le = LabelEncoder()
y  = le.fit_transform(adata.obs['cell_type'])
print(f'Cell types: {le.classes_}')

# Train/test split — fix this split for all methods
indices = np.arange(len(y))
train_idx, test_idx = train_test_split(
    indices, test_size=0.2, random_state=42, stratify=y
)
y_train = y[train_idx]
y_test  = y[test_idx]
print(f'Train: {len(train_idx)}, Test: {len(test_idx)}')

## Step 3: Method 1 — PCA

In [ ]:
print('Running PCA...')
t0 = time.time()

sc.tl.pca(adata, n_comps=50, use_highly_variable=True)
X_pca = adata.obsm['X_pca']

pca_time = time.time() - t0
print(f'PCA done in {pca_time:.1f}s  |  Shape: {X_pca.shape}')

knn = KNeighborsClassifier(n_neighbors=15, metric='cosine', n_jobs=-1)
knn.fit(X_pca[train_idx], y_train)
y_pred_pca = knn.predict(X_pca[test_idx])

acc_pca = accuracy_score(y_test, y_pred_pca)
f1_pca  = f1_score(y_test, y_pred_pca, average='macro')
print(f'PCA + kNN  |  Accuracy: {acc_pca:.3f}  |  Macro-F1: {f1_pca:.3f}')

## Step 4: Method 2 — scVI

In [ ]:
print('Running scVI...')
t0 = time.time()

adata_scvi = adata.copy()
adata_scvi.X = adata_scvi.layers['counts']

scvi.model.SCVI.setup_anndata(
    adata_scvi,
    layer='counts',
    batch_key='batch'
)
model_scvi = scvi.model.SCVI(adata_scvi, n_latent=20, n_layers=2)
model_scvi.train(max_epochs=200, early_stopping=True,
                  early_stopping_patience=20, batch_size=256)

X_scvi = model_scvi.get_latent_representation()
adata.obsm['X_scVI'] = X_scvi

scvi_time = time.time() - t0
print(f'scVI done in {scvi_time:.1f}s  |  Shape: {X_scvi.shape}')

knn_scvi = KNeighborsClassifier(n_neighbors=15, metric='cosine', n_jobs=-1)
knn_scvi.fit(X_scvi[train_idx], y_train)
y_pred_scvi = knn_scvi.predict(X_scvi[test_idx])

acc_scvi = accuracy_score(y_test, y_pred_scvi)
f1_scvi  = f1_score(y_test, y_pred_scvi, average='macro')
print(f'scVI + kNN  |  Accuracy: {acc_scvi:.3f}  |  Macro-F1: {f1_scvi:.3f}')

## Step 5: Method 3 — Geneformer

> **Switch to GPU runtime before running this cell**  
> Runtime > Change runtime type > T4 GPU

In [ ]:
# Note: use the current stable Geneformer release
# Check https://huggingface.co/ctheodoris/Geneformer for the latest model
from geneformer import TranscriptomeTokenizer, EmbExtractor
import os

GENEFORMER_MODEL = 'ctheodoris/Geneformer'  # update to latest stable

print('Tokenising for Geneformer...')
t0 = time.time()

os.makedirs('gf_input',  exist_ok=True)
os.makedirs('gf_tokens', exist_ok=True)
os.makedirs('gf_embs',   exist_ok=True)

# Save as loom for tokenisation
adata_gf = adata.copy()
adata_gf.X = adata_gf.layers['counts'].astype(int)
adata_gf.write_loom('gf_input/pbmc.loom')

tk = TranscriptomeTokenizer(
    {'cell_type': 'cell_type', 'batch': 'batch'},
    nproc=2
)
tk.tokenize_data('gf_input/', 'gf_tokens/', 'pbmc', file_format='loom')

print('Extracting Geneformer embeddings...')
embex = EmbExtractor(
    model_type='Pretrained', num_classes=0,
    emb_mode='cell', cell_emb_style='mean_pool',
    emb_layer=-1, emb_label=['cell_type'],
    forward_batch_size=100, nproc=2
)
embs = embex.extract_embs(GENEFORMER_MODEL,
                           'gf_tokens/pbmc.dataset',
                           'gf_embs/', 'pbmc')

X_gf = embs.values
adata.obsm['X_geneformer'] = X_gf
gf_time = time.time() - t0

print(f'Geneformer done in {gf_time:.1f}s  |  Shape: {X_gf.shape}')

knn_gf = KNeighborsClassifier(n_neighbors=15, metric='cosine', n_jobs=-1)
knn_gf.fit(X_gf[train_idx], y_train)
y_pred_gf = knn_gf.predict(X_gf[test_idx])

acc_gf = accuracy_score(y_test, y_pred_gf)
f1_gf  = f1_score(y_test, y_pred_gf, average='macro')
print(f'Geneformer + kNN  |  Accuracy: {acc_gf:.3f}  |  Macro-F1: {f1_gf:.3f}')

## Step 6: Final Comparison

In [ ]:
results = pd.DataFrame({
    'Method':   ['PCA + kNN', 'scVI + kNN', 'Geneformer + kNN'],
    'Accuracy': [acc_pca,  acc_scvi,  acc_gf],
    'Macro-F1': [f1_pca,   f1_scvi,   f1_gf],
    'Time (s)': [pca_time, scvi_time, gf_time]
}).sort_values('Macro-F1', ascending=False)

print('\n' + '='*55)
print('FINAL RESULTS: Did the Foundation Model Actually Help?')
print('='*55)
print(results.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
colors = ['#38bdf8', '#4ade80', '#a78bfa']
axes[0].bar(results['Method'], results['Macro-F1'], color=colors)
axes[0].set_ylabel('Macro-F1')
axes[0].set_title('Classification Performance')
axes[0].tick_params(axis='x', rotation=15)
axes[1].bar(results['Method'], results['Time (s)'], color=colors)
axes[1].set_ylabel('Runtime (seconds)')
axes[1].set_title('Computational Cost')
axes[1].tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()

## Step 7: UMAP Visualisation

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
methods = [('PCA','X_pca'), ('scVI','X_scVI'), ('Geneformer','X_geneformer')]

for col, (name, key) in enumerate(methods):
    if key not in adata.obsm:
        print(f'Skipping {name} — embedding not found')
        continue
    sc.pp.neighbors(adata, use_rep=key, n_neighbors=15, key_added=f'nn_{name}')
    sc.tl.umap(adata, neighbors_key=f'nn_{name}', key_added=f'umap_{name}')
    sc.pl.embedding(adata, basis=f'umap_{name}', color='cell_type',
                    ax=axes[0,col], show=False, title=f'{name} — Cell Type')
    sc.pl.embedding(adata, basis=f'umap_{name}', color='batch',
                    ax=axes[1,col], show=False, title=f'{name} — Donor')

plt.tight_layout()
plt.show()

## Your Answer

Fill in the template below with your results:

```
RESULTS:
  PCA Macro-F1:         ___
  scVI Macro-F1:        ___
  Geneformer Macro-F1:  ___

CONCLUSION:
  Did Geneformer outperform scVI? By how much?
  Was the performance gain worth the extra runtime?
  Which method would you use in a real analysis?
  What surprised you most?
```

---
**BioSkills Lab** — bioskillslab.dev